## Table: `03_gold`.d_restaurant_reviews

This table provides aggregated review and sentiment statistics for each restaurant. It is designed to support analytics and reporting on restaurant performance and customer feedback.

- **restaurant_id**: Unique identifier for each restaurant (Primary Key).
- **restaurant_name**: Name of the restaurant.
- **city**: City where the restaurant is located.
- **total_reviews**: Total number of reviews received.
- **avg_rating**: Average rating across all reviews (DECIMAL(3,2)).
- **rating_5_count**: Number of 5-star ratings.
- **rating_4_count**: Number of 4-star ratings.
- **rating_3_count**: Number of 3-star ratings.
- **rating_2_count**: Number of 2-star ratings.
- **rating_1_count**: Number of 1-star ratings.
- **sentiment_positive_count**: Count of reviews with positive sentiment.
- **sentiment_neutral_count**: Count of reviews with neutral sentiment.
- **sentiment_negative_count**: Count of reviews with negative sentiment.

This table enables insights into restaurant quality, customer satisfaction, and sentiment trends.

In [0]:
df_restaurants = spark.read.table("databricksproject_ws.`02_silver`.dim_restaurants")
df_reviews = spark.read.table("databricksproject_ws.`02_silver`.fact_reviews")
display(df_restaurants.limit(1))
display(df_reviews.limit(1))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

# Read Silver Tables
df_restaurants = spark.read.table("databricksproject_ws.`02_silver`.dim_restaurants").filter(col("__END_AT").isNull())
df_reviews = spark.read.table("databricksproject_ws.`02_silver`.fact_reviews")

# Aggregate Review Statistics
df_review_stats = (
    df_reviews
        .groupBy("restaurant_id")
        .agg(
            count("review_id").alias("total_reviews"),
            round(avg("rating"), 2).cast(DecimalType(3, 2)).alias("avg_rating"),

            sum(when(col("rating") == 5, 1).otherwise(0)).alias("rating_5_count"),
            sum(when(col("rating") == 4, 1).otherwise(0)).alias("rating_4_count"),
            sum(when(col("rating") == 3, 1).otherwise(0)).alias("rating_3_count"),
            sum(when(col("rating") == 2, 1).otherwise(0)).alias("rating_2_count"),
            sum(when(col("rating") == 1, 1).otherwise(0)).alias("rating_1_count"),

            sum(when(col("sentiment") == "positive", 1).otherwise(0)).alias("sentiment_positive_count"),
            sum(when(col("sentiment") == "neutral", 1).otherwise(0)).alias("sentiment_neutral_count"),
            sum(when(col("sentiment") == "negative", 1).otherwise(0)).alias("sentiment_negative_count")
        )
)

# Build Gold Dataset
df_restaurant_reviews = (
    df_restaurants
        .join(df_review_stats, "restaurant_id", "left")
        .select(
            col("restaurant_id"),
            col("name").alias("restaurant_name"),
            col("city"),

            coalesce(col("total_reviews"), lit(0)).cast("bigint").alias("total_reviews"),
            coalesce(col("avg_rating"), lit(0)).cast(DecimalType(3, 2)).alias("avg_rating"),

            coalesce(col("rating_5_count"), lit(0)).cast("bigint").alias("rating_5_count"),
            coalesce(col("rating_4_count"), lit(0)).cast("bigint").alias("rating_4_count"),
            coalesce(col("rating_3_count"), lit(0)).cast("bigint").alias("rating_3_count"),
            coalesce(col("rating_2_count"), lit(0)).cast("bigint").alias("rating_2_count"),
            coalesce(col("rating_1_count"), lit(0)).cast("bigint").alias("rating_1_count"),

            coalesce(col("sentiment_positive_count"), lit(0)).cast("bigint").alias("sentiment_positive_count"),
            coalesce(col("sentiment_neutral_count"), lit(0)).cast("bigint").alias("sentiment_neutral_count"),
            coalesce(col("sentiment_negative_count"), lit(0)).cast("bigint").alias("sentiment_negative_count")
        )
)

display(df_restaurant_reviews.limit(5))